### Libraries

In [5]:
import json
import numpy as np
import pandas as pd
from datetime import datetime
import folium
import math
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt
import overpy
from geopy.distance import geodesic
from geopy import Point

### Load the excel file

In [132]:
file_path = "D:\OVGU\AMS project\GIT\AMS_ShipTraj\physics based model\Rheinhafen Data.xlsx"
df = pd.read_excel(file_path, header=None)
print(df.head())

time_series = df[1]
json_series = df[2]

records = []

for time_val, json_str in zip(time_series, json_series):
    try:
        # Parse the valid JSON string directly
        data_dict = json.loads(json_str)
        data_dict["time"] = str(time_val)
        records.append(data_dict)
    except Exception as e:
        print(f"Error processing record at time {time_val}: {e}")

# Optional: print first record
print(records[0])

for i in range(5):
    print(records[i])

         0                       1  \
0  3730963 2025-03-09 00:00:00.397   
1  3730964 2025-03-09 00:00:04.230   
2  3730965 2025-03-09 00:00:06.277   
3  3730966 2025-03-09 00:00:07.531   
4  3730967 2025-03-09 00:00:10.537   

                                                   2  
0  {"msg_type": 1, "repeat": 0, "mmsi": 244660361...  
1  {"msg_type": 1, "repeat": 0, "mmsi": 211576360...  
2  {"msg_type": 1, "repeat": 0, "mmsi": 211478290...  
3  {"msg_type": 1, "repeat": 0, "mmsi": 211632780...  
4  {"msg_type": 1, "repeat": 0, "mmsi": 244660361...  
{'msg_type': 1, 'repeat': 0, 'mmsi': 244660361, 'status': 0, 'turn': 0.0, 'speed': 9.0, 'accuracy': True, 'lon': 8.262617, 'lat': 48.981312, 'course': 46.9, 'heading': 47, 'second': 57, 'maneuver': 1, 'spare_1': 'AA==', 'raim': False, 'radio': 2240, 'time': '2025-03-09 00:00:00.397000'}
{'msg_type': 1, 'repeat': 0, 'mmsi': 244660361, 'status': 0, 'turn': 0.0, 'speed': 9.0, 'accuracy': True, 'lon': 8.262617, 'lat': 48.981312, 'course': 46

### Take a particular mmsi id

In [133]:
from datetime import datetime

# Step 1: Filter by MMSI
target_mmsi = 244660361
filtered_records = [record for record in records if record['mmsi'] == target_mmsi]

# Step 2: Sort by time (convert 'time' to datetime during sorting)
sorted_records = sorted(
    filtered_records,
    key=lambda x: datetime.fromisoformat(x['time'])  # Converts '2025-03-09 00:00:00.397000' to datetime
)

# Optional: Show result
for rec in sorted_records[:3]:
    print(rec)

{'msg_type': 1, 'repeat': 0, 'mmsi': 244660361, 'status': 0, 'turn': 0.0, 'speed': 9.0, 'accuracy': True, 'lon': 8.262617, 'lat': 48.981312, 'course': 46.9, 'heading': 47, 'second': 57, 'maneuver': 1, 'spare_1': 'AA==', 'raim': False, 'radio': 2240, 'time': '2025-03-09 00:00:00.397000'}
{'msg_type': 1, 'repeat': 0, 'mmsi': 244660361, 'status': 0, 'turn': 0.0, 'speed': 9.0, 'accuracy': True, 'lon': 8.263082, 'lat': 48.981605, 'course': 46.1, 'heading': 46, 'second': 7, 'maneuver': 1, 'spare_1': 'AA==', 'raim': False, 'radio': 2234, 'time': '2025-03-09 00:00:10.537000'}
{'msg_type': 1, 'repeat': 0, 'mmsi': 244660361, 'status': 0, 'turn': 0.0, 'speed': 9.1, 'accuracy': True, 'lon': 8.263542, 'lat': 48.981902, 'course': 45.9, 'heading': 46, 'second': 17, 'maneuver': 1, 'spare_1': 'AA==', 'raim': False, 'radio': 16384, 'time': '2025-03-09 00:00:20.351000'}


### Load Line of Sight

In [92]:
json_path = "D:/OVGU/AMS project/GIT/AMS_ShipTraj/physics based model/sorted_centerline.json"

with open(json_path, 'r') as f:
    line_of_sight = json.load(f)

# Optional: preview a few points
print(f"Total points in line_of_sight: {len(line_of_sight)}")
print("First 3 points:", line_of_sight[:3])

Total points in line_of_sight: 744
First 3 points: [[48.884527000000006, 8.133417], [48.884798, 8.133690999999999], [48.885025, 8.133922]]


### Line of Sight reverse

In [98]:
line_of_sight_reverse = line_of_sight[::-1]

# Optional: Preview the reversed list
print("First 3 points of line_of_sight_reverse:", line_of_sight_reverse[:3])


First 3 points of line_of_sight_reverse: [[49.3941995, 8.498329], [49.394199, 8.498342000000001], [49.3939055, 8.499148]]


### Load the parameters

In [232]:
class ShipParameters:
    def __init__(self, initial_x, initial_y, heading_deg):
        # Physical constants
        self.rho_water = 1000  # kg/m³

        # Propeller parameters
        self.D = 3.0            # diameter [m]
        self.KT = 0.3           # thrust coefficient
        self.target_n_rpm = 150  # target rpm
        self.n_rpm = 0.0         # start from 0
        self.n = 0.0             # rev/s, will update during simulation

        # Rudder parameters
        self.AR = 15            # area [m²]
        self.CL = 0.8           # lift coefficient
        self.CD_rudder = 0.02   # drag coefficient
        self.delta_deg = 0    # rudder angle in degrees
        self.delta = np.radians(self.delta_deg)

        # Initial state
        self.u = 9.0            # surge velocity
        self.v = 0.0            # sway velocity
        self.r = 0.0            # yaw rate

        self.x = initial_x      # user input
        self.y = initial_y
        self.psi = np.radians(heading_deg)

        # Ship parameters
        self.mass = 6_500_000   # kg
        self.Iz = 4e8           # yaw moment of inertia
        self.xR = 60.0          # rudder distance from CG [m]

        # Drag
        self.S = 4000           # wetted surface [m²]
        self.CD = 0.003         # hull drag coeff

        self.tp = 0.15          # Thrust deduction factor
        self.w = 0.15           # Wake fraction


        # Simulation
        self.dt = 1           # time step [s]


### Calculate the forces

In [233]:
def calculate_J(params):
    denom = params.n * params.D
    if denom == 0:
        return 0.0  # Prevent divide by zero
    J = params.u * (1 - params.w) / denom
    return min(J, 5.0)  # clamp extreme cases


def KT_of_J(J):
    # Example quadratic fit
    a, b, c = 1.333, -0.8, -0.0667
    KT = a * J**2 + b * J + c
    if np.isnan(KT) or np.isinf(KT):
        return 0.0
    return max(0.0, min(KT, 0.4))  # clamp to realistic range


def calculate_propeller_force(params):
    J = calculate_J(params)
    KT = KT_of_J(J)
    TP = params.rho_water * params.n**2 * params.D**4 * KT
    XP = (1 - params.tp) * TP  # Surge contribution
    return XP

def calculate_drag_force(params):
    return 0.5 * params.rho_water * params.S * params.CD * params.u**2

def CL_of_delta(delta_rad):
    CL_slope = 6.0  # Lift slope (per radian)
    return CL_slope * delta_rad

def CD_of_delta(delta_rad):
    CD0 = 0.02
    k = 2.5
    return CD0 + k * delta_rad**2

def calculate_rudder_forces(params):
    params.delta = np.clip(params.delta, -np.radians(35), np.radians(35))

    UR = params.u
    delta = params.delta

    CL = CL_of_delta(delta)
    CD_rudder = CD_of_delta(delta)

    LR = 0.5 * params.rho_water * params.AR * UR**2 * CL
    DR = 0.5 * params.rho_water * params.AR * UR**2 * CD_rudder

    XR = -DR * np.cos(delta) + LR * np.sin(delta)
    YR = -DR * np.sin(delta) - LR * np.cos(delta)
    NR = YR * params.xR

    return XR, YR, NR

### Calculating the heading

In [234]:
def calculate_bearing(pointA, pointB):
    """
    Returns bearing from pointA to pointB (in degrees)
    """
    lat1, lon1 = map(math.radians, pointA)
    lat2, lon2 = map(math.radians, pointB)

    d_lon = lon2 - lon1

    x = math.sin(d_lon) * math.cos(lat2)
    y = math.cos(lat1)*math.sin(lat2) - math.sin(lat1)*math.cos(lat2)*math.cos(d_lon)

    bearing = math.atan2(x, y)
    return (math.degrees(bearing) + 360) % 360


### Simulation

In [235]:
def simulate_motion(params, simulation_path, duration_minutes):

    # Parameters
    k = 0.3985  # deg heading change per deg rudder per second
    max_rudder_deg = 35
    ramp_duration_sec = 60  # propeller ramp-up duration

    dt = params.dt
    steps = int(duration_minutes * 60 / dt)

    # History trackers
    x_hist, y_hist = [params.x], [params.y]

    # Reference point tracking
    ref_idx = 0
    current_ref = simulation_path[ref_idx]
    ref_distance = geodesic((params.x, params.y), current_ref).meters

    for step in range(steps):
        time_elapsed_sec = step * dt

        # --- Ramp-up Propeller Speed ---
        if time_elapsed_sec <= ramp_duration_sec:
            params.n_rpm = (params.target_n_rpm * time_elapsed_sec) / ramp_duration_sec
        else:
            params.n_rpm = params.target_n_rpm
        params.n = params.n_rpm / 60.0  # rev/s

        # --- Calculate required heading ---
        current_pos = (params.x, params.y)
        bearing = calculate_bearing(current_pos, current_ref)
        required_heading = bearing % 360
        current_heading = np.degrees(params.psi) % 360

        # --- Compute heading error and normalize to [-180, 180] ---
        heading_error = required_heading - current_heading

        # --- Rudder angle control ---
        if heading_error > 6 or heading_error < -6:
            rudder_angle = -1 * heading_error / k
            rudder_angle = np.clip(rudder_angle, -max_rudder_deg, max_rudder_deg)
            params.delta = np.radians(rudder_angle)
        #print(heading_error, rudder_angle, params.delta)

        # --- Calculate Forces ---
        Fp = calculate_propeller_force(params)
        Fd = calculate_drag_force(params)
        XR, YR, NR = calculate_rudder_forces(params)

        # --- Update surge, sway, and yaw ---
        a_surge = (Fp - Fd + XR) / params.mass
        params.u = np.clip(params.u + a_surge * dt, 0.0, 12.0)

        a_sway = YR / params.mass
        params.v += a_sway * dt

        r_prev = params.r 
        r_dot = NR / params.Iz
        params.r += r_dot * dt
        if params.r != r_prev:
            params.psi += params.r * dt

        # --- Position update (dx, dy in meters) ---
        dx = (params.u * np.cos(params.psi) - params.v * np.sin(params.psi)) * dt
        dy = (params.u * np.sin(params.psi) + params.v * np.cos(params.psi)) * dt

        origin = Point(params.x, params.y)
        move_dist = math.hypot(dx, dy)
        move_bearing = math.degrees(math.atan2(dx, dy))
        new_point = geodesic(meters=move_dist).destination(origin, move_bearing)

        params.x = new_point.latitude
        params.y = new_point.longitude

        x_hist.append(params.x)
        y_hist.append(params.y)

        # --- Check if we need to move to next reference point ---
        distance_to_ref = geodesic((params.x, params.y), current_ref).meters
        if distance_to_ref < ref_distance / 2 and ref_idx + 1 < len(simulation_path):
            prev_ref = current_ref
            ref_idx += 1
            current_ref = simulation_path[ref_idx]
            ref_distance = geodesic(prev_ref, current_ref).meters

    return x_hist, y_hist, params


### Calculate the position

In [236]:
initial_x = 48.981312
initial_y = 8.262617
initial_heading_deg = 40
final_x = 48.99783768657771
final_y = 8.287111544559066



In [237]:
use_reverse = (final_x - initial_x) < 0
source_path = line_of_sight_reverse if use_reverse else line_of_sight


x_min = min(initial_x, final_x)
x_max = max(initial_x, final_x)


simulation_path = [pt for pt in source_path if x_min <= pt[0] <= x_max]

# Optional: Show number of points and first few
print(f"Direction: {'reverse' if use_reverse else 'forward'}")
print(f"Total points in simulation_path: {len(simulation_path)}")
print("First 3 points:", simulation_path[:3])

Direction: forward
Total points in simulation_path: 26
First 3 points: [[48.9816785, 8.263064], [48.982336, 8.264011], [48.9830055, 8.2649655]]


In [238]:
# Reduce the reference path
reduced_path = simulation_path[::2]  # Every alternate point

# Initialize parameters
params = ShipParameters(initial_x, initial_y, initial_heading_deg)

# Run simulation
x_traj, y_traj, final_params = simulate_motion(params, reduced_path, duration_minutes=10)



# Print final result
"""print(f"Final Position: ({final_params.x:.6f}, {final_params.y:.6f})")
print(f"Final Speed: {final_params.u:.2f} m/s")"""
print(f"Final Heading: {np.degrees(final_params.psi)%360:.2f}°")
print(final_params.r)

Final Heading: 99.75°
-179.4047864357013


### Ploting the position using Folium

In [239]:
# 1. Define start location for map center
start_location = [x_traj[0], y_traj[0]]
m = folium.Map(location=start_location, zoom_start=14)

# 2. Create trajectory path (ship motion path)
trajectory = list(zip(x_traj, y_traj))

# 3. Add simulated trajectory to map
folium.PolyLine(
    locations=trajectory,
    color="blue",
    weight=3,
    opacity=0.7,
    tooltip="Simulated Ship Path"
).add_to(m)

# 4. Add start and end markers for trajectory
folium.Marker(location=trajectory[0], popup="Start", icon=folium.Icon(color='green')).add_to(m)
folium.Marker(location=trajectory[-1], popup="End", icon=folium.Icon(color='red')).add_to(m)

# 5. Add reference path (simulation_path)
folium.PolyLine(
    locations=simulation_path,
    color="purple",
    weight=2,
    opacity=0.7,
    tooltip="Reference Path (simulation_path)"
).add_to(m)

# 6. Optionally: add marker at each reference point
for idx, point in enumerate(simulation_path):
    folium.CircleMarker(
        location=point,
        radius=2,
        color="purple",
        fill=True,
        fill_opacity=0.7,
        popup=f"Ref {idx}"
    ).add_to(m)

# 7. Display the map
m